## Projeto Polars
### Análise base de dados Varejo e Atributos Outbox 

#### Metodologia CRISP-DM

**Entendimento do negócio** <br>
**Entendimento dos dados** (Estamos aqui) <br>
**Preparação dos dados** (Desejamos finalizar hoje)<br>
**Modelagem** <br>
**Avaliação** <br>
**Implementação** <br>

#### Dependências

In [15]:
from pathlib import Path
from datetime import datetime, date, timedelta

import polars as pl
import polars.selectors as cs
from polars.datatypes import UInt16, Utf8, Float32, Date
from polars.exceptions import (
    SchemaError,
    ComputeError,
    ColumnNotFoundError,
    PolarsError,
)

import duckdb as dd


#### Estrutura de arquivos
```
RaizProjeto
|____ data
     |____ raw
     |____ staging
           |____ base_amostrada.parquet
           |____ base_tratada.parquet
     |____ prod
           |____ modelo
           |____ base_premodelagem.parquet
|____ doc
     | requisitos.txt
     | metricas.txt
|____ src
     |____ config
           |____ caminhos.yaml
     |____ constantes
           |____ safras.yaml
     |____ tratamento_dados
           |____ limpeza_nulos.py
           |____ limpeza_preenchimento.py
           |____ preenche_valores.py
           |____ valores_extremos.py
           |____ categorizacao.py
     |____ modelo
           |____ classificador_xgb.py
           |____ logistica_simples.py
     |____ avaliacao
           |____ metricas_avaliacao.py
           |____ comparativo_modelos.py
     | main.py
|____ tests
|____ venv
```

#### Leitura das bases

Famílias de Métodos: <br>
**scan_** <br>
Utilizados a leitura de um arquivo/tabela para um grafo/plano de consulta. Não é indicado para fluxos de trabalho onde seja necessário conhecer os valores dos dados. Porém, o método *.collect* pode ser oportunamente chamado quando necessário e o datafrale liberado posteriormente chamando o método *.lazy*.

**read_** <br>
Utilizados a leitura de um arquivo/tabela para um dataframe colunar. 

Formatos de arquivos e outros
- Parquet
- CSV
- Avro
- Delta
- JSON

Interoperabilidade com outras engines/bibliotecas:<br>
**Pandas** <br>

Um dataframe Polars pode ser convertido de maneira simples. Aplicar o método *.to_pandas()* converte a o dataframe Polars em um dataframe Pandas. Alguns métodos do polars fazem a serialização para tipos numpy para calcular algumas funções estatísticas

```python
import polars as pl
import duckdb as dd

df = pl.DataFrame({"foo": [1, 2, 3], "bar": [4, 5, 6]})
pandas_df = df.to_pandas()
print(pandas_df)
``` 

**Duckdb** <br>
Utilizar um dataframe Polars com o DuckDB é extremamente simples, uma vez que este pode ser acessado de forma similar ao acessar uma tabela em um banco de dados convencional.

```python
import polars as pl
import duckdb as dd

df = pl.DataFrame({"foo": [1, 2, 3], "bar": [4, 5, 6]})
dd.sql(
    """
        select * from df
    """
).show()
```
**Pyspark**<br>
É pouco provável precisar deste tipo de conversão, mas uma forma possível e não performática é converter o dataframe em uma lista de dicionários, iterar sobre os seus itens e construir um dataframe Spark através do método createDataFrame
```python

import polars as pl
from pyspark.sql import SparkSession

df = pl.DataFrame({"foo": [1, 2, 3], "bar": [4, 5, 6]})

os.environ["SPARK_HOME"] = "D:/spark"
os.environ["HADOOP_HOME"] = "D:/spark/hadoop"

spark = SparkSession.builder.appName("Teste").getOrCreate()
spark.createDataFrame({x:y.to_list() for x,y in df.to_dict().items()})
```

Outras situações importantes podem envolver transformar um dataframe em componentes serializáveis (dicionários) que podem posteriormente serem convertidos em json e transitados via uma API HTTP.

```python
df = pl.DataFrame({"foo": [1, 2, 3], "bar": [4, 5, 6]})
lista_dicionarios = df.to_dicts()

print(lista_dicionarios)
```

In [20]:
caminho_base = Path("C:/Users/RenatoFileto/curso_bigdata/ModuloPython/data/raw")
caminho_base_csv = caminho_base.joinpath("base_semana02.csv")

grafo_base_atributos = pl.scan_csv(
    caminho_base_csv,
    has_header=True,
    separator=";",
    infer_schema_length=100000,
    encoding="utf8",
    cache=False,
)


OrderedDict([('id', Int64), ('amostra', String), ('foto', String), ('conceito', Int64), ('var_1', Int64), ('var_2', Int64), ('var_3', Float64), ('var_4', Float64), ('var_5', Int64), ('var_6', Int64), ('var_7', Float64), ('var_8', Float64), ('var_9', Int64), ('var_10', Int64), ('var_11', Int64), ('var_12', Int64), ('var_13', Int64), ('var_14', Int64), ('var_15', Int64), ('var_16', Float64), ('var_17', Float64), ('var_18', Float64), ('var_19', Float64), ('var_20', Float64), ('var_21', String), ('var_22', String), ('var_23', String), ('var_24', String), ('var_25', String), ('var_26', String), ('var_27', String), ('var_28', String), ('var_29', String), ('var_30', String), ('var_31', Int64), ('var_32', Int64), ('var_33', Int64), ('var_34', Int64), ('var_35', Int64), ('var_36', Float64), ('var_37', Float64), ('var_38', Float64), ('var_39', Float64), ('var_40', Float64), ('var_41', String), ('var_42', String), ('var_43', String), ('var_44', String), ('var_45', String), ('var_46', String), ('v

In [25]:
colunas_float = cs.float()
print(len(grafo_base_atributos.select(colunas_float).columns))


103


#### Estratégias de sink e collect

Em determinadas situações pode ser relevante utilizar um grafo de consulta ao invés de um dataframe, mesmo em situações que demandem o conhecimento dos valores dos dados. Nestes momentos podemos coletar apenas porções do dataframe.

Se a situação demandar que uma grande quantidade de dados seja coletada e que haja a inteção/necessidade de converter o DataFrame em um LazyFrame, é mais provável que uma estratégia de sink seja mais produtiva, ou seja, podemos gravar os dados no disco e efetuar uma nova leitura posterior, utilizando algum método da família scan.

In [40]:
dicionario_meses_pt_br = {
    "jan": 1,
    "fev": 2,
    "mai": 5,
    "jul": 7,
    "set": 9,
    "dez": 12,
}

grafo_base_atributos = (
    grafo_base_atributos.with_columns(
        split1=pl.col("foto").str.extract_all(r"\D+").list.first(),
        split2=pl.col("foto").str.extract_all(r"\d+").list.first(),
    )
    .with_columns(trat_foto=pl.col("split1").str.to_lowercase().str.strip_chars(" "))
    .with_columns(
        trat_mes=pl.col("trat_foto").replace(dicionario_meses_pt_br),
        trat_ano=(pl.col("split2").cast(pl.Int16) + 2000),
    )
    .with_columns(
        trat_data_ref=pl.date(pl.col("trat_ano"), pl.col("trat_mes"), 1).dt.month_end()
    )
)

lista_datas_referencia: pl.Series = (
    grafo_base_atributos.select("trat_data_ref").unique().collect().to_series()
)

lista_resultados_por_data: list[pl.LazyFrame] = []
for data in lista_datas_referencia:
    lista_resultados_por_data.append(
        grafo_base_atributos.select("foto", "trat_data_ref").with_columns(
            tmp_data_ref=(pl.col("trat_data_ref") - pl.lit(data)).dt.total_days()
        )
    )

base_empilhada = pl.concat(lista_resultados_por_data, how="vertical", parallel=True)
print(base_empilhada.collect().head(10))


C:\Users\RenatoFileto\AppData\Local\Temp\ipykernel_5696\2838721463.py:33: DeprecationWarning: `days` is deprecated. It has been renamed to `total_days`.
  tmp_data_ref=(pl.col("trat_data_ref") - pl.lit(data)).dt.days()


shape: (10, 3)
┌───────┬───────────────┬──────────────┐
│ foto  ┆ trat_data_ref ┆ tmp_data_ref │
│ ---   ┆ ---           ┆ ---          │
│ str   ┆ date          ┆ i64          │
╞═══════╪═══════════════╪══════════════╡
│ set22 ┆ 2022-09-30    ┆ 61           │
│ mai22 ┆ 2022-05-31    ┆ -61          │
│ set22 ┆ 2022-09-30    ┆ 61           │
│ jul22 ┆ 2022-07-31    ┆ 0            │
│ dez21 ┆ 2021-12-31    ┆ -212         │
│ fev23 ┆ 2023-02-28    ┆ 212          │
│ jul22 ┆ 2022-07-31    ┆ 0            │
│ dez21 ┆ 2021-12-31    ┆ -212         │
│ fev22 ┆ 2022-02-28    ┆ -153         │
│ fev23 ┆ 2023-02-28    ┆ 212          │
└───────┴───────────────┴──────────────┘


In [74]:
lista_resultados: list[pl.LazyFrame] = []
for data in lista_datas_referencia:
    resultado_intermediario = grafo_base_atributos.filter(
        (pl.col("trat_data_ref") < pl.lit(data))
        & (pl.col("trat_data_ref") >= pl.col("trat_data_ref").dt.offset_by(by="-3mo"))
    )
    print(
        resultado_intermediario.select(
            pl.lit(data),
            pl.col("trat_data_ref").min().alias("data_ref_min"),
            pl.col("trat_data_ref").max().alias("data_ref_max"),
        ).collect()
    )
    lista_resultados.append(resultado_intermediario)

# base_empilhada_offset = pl.concat(lista_resultados, how="vertical", parallel=True)


shape: (1, 3)
┌────────────┬──────────────┬──────────────┐
│ literal    ┆ data_ref_min ┆ data_ref_max │
│ ---        ┆ ---          ┆ ---          │
│ date       ┆ date         ┆ date         │
╞════════════╪══════════════╪══════════════╡
│ 2022-07-31 ┆ 2021-12-31   ┆ 2022-05-31   │
└────────────┴──────────────┴──────────────┘
shape: (1, 3)
┌────────────┬──────────────┬──────────────┐
│ literal    ┆ data_ref_min ┆ data_ref_max │
│ ---        ┆ ---          ┆ ---          │
│ date       ┆ date         ┆ date         │
╞════════════╪══════════════╪══════════════╡
│ 2023-02-28 ┆ 2021-12-31   ┆ 2022-09-30   │
└────────────┴──────────────┴──────────────┘
shape: (1, 3)
┌────────────┬──────────────┬──────────────┐
│ literal    ┆ data_ref_min ┆ data_ref_max │
│ ---        ┆ ---          ┆ ---          │
│ date       ┆ date         ┆ date         │
╞════════════╪══════════════╪══════════════╡
│ 2022-05-31 ┆ 2021-12-31   ┆ 2022-02-28   │
└────────────┴──────────────┴──────────────┘
shape: (1, 3)

Dicas para escolher bibliotecas

Até 500 mil linhas -> pandas
Até 100_000_000 linhas -> polars/dask/duckdb
Daí em diante Spark

#### Remoção de colunas e linhas nulas

Métodos:
- **drop_null**

In [ ]:
contagem_nulos_df = grafo_base_atributos.null_count().collect()
tamanho_df = grafo_base_atributos.select("id").count().collect().to_series()

perc_nulos_df = contagem_nulos_df / tamanho_df


In [39]:
dicionario_nulos = perc_nulos_df.to_dict()

trat_grafo_base_atributos = grafo_base_atributos
lista_colunas_removidas: list[str] = []
total_colunas_removidas: int = 0

for coluna in grafo_base_atributos.columns:
    if dicionario_nulos[coluna][0] >= 0.95:
        trat_grafo_base_atributos = trat_grafo_base_atributos.drop(coluna)
        total_colunas_removidas += 1
        lista_colunas_removidas.append(coluna)

print(total_colunas_removidas)

try:
    trat_grafo_base_atributos.sink_parquet(
        caminho_base.joinpath("trat_base_nulos.parquet")
    )
except PolarsError as e:
    print(e)
    print("Processo Finalizado com erro")
except Exception as e:
    print(e)
except FileNotFoundError as e:
    print(e)
else:
    print("Processo Finalizado")


370
Processo Finalizado


In [ ]:
df = pl.DataFrame({"foo": [1, None, 3], "bar": [4, 5, 6]})
print(df)
print(df.drop_nulls("bar"))


#### Inputação de valores

Métodos Polars:
- **fill_null** e estratégias *foward*, *backward* e *min, max, mean e median*
- **fill_nan**

Existem outras formas e métodos mais sofisticados de lidar com estes valores.  Uma delas é aplicar um algoritmo K-Nearest Neighbors e utilizar a média dos valores do vizinhos. Também é possível utilizar um algoritmo de regressão para determinar o valor a ser preenchido.


In [54]:
df = pl.DataFrame({"foo": [1, None, 3], "bar": [4, 5, float("nan")]})

df1 = df.fill_null(strategy="mean")
print(df1)

df2 = df.fill_nan(99999999)
print(df2)


shape: (3, 2)
┌─────┬─────┐
│ foo ┆ bar │
│ --- ┆ --- │
│ i64 ┆ f64 │
╞═════╪═════╡
│ 1   ┆ 4.0 │
│ 2   ┆ 5.0 │
│ 3   ┆ NaN │
└─────┴─────┘
shape: (3, 2)
┌──────┬─────────────┐
│ foo  ┆ bar         │
│ ---  ┆ ---         │
│ i64  ┆ f64         │
╞══════╪═════════════╡
│ 1    ┆ 4.0         │
│ null ┆ 5.0         │
│ 3    ┆ 9.9999999e7 │
└──────┴─────────────┘


#### Remoção de colunas que possuam poucos elementos distintos (Exercício)

#### Lag e Lead

In [56]:
print(trat_grafo_base_atributos.limit(10).collect())

trat_grafo_base_atributos = trat_grafo_base_atributos.with_columns(
    cmp_data_ref=(
        pl.col("trat_data_ref") - pl.col("trat_data_ref").shift(-1).over("amostra")
    ).dt.total_days(),
    cmp_data_ref_max=(
        pl.col("trat_data_ref") - pl.col("trat_data_ref").max().over("amostra")
    ).dt.total_days(),
)

print(trat_grafo_base_atributos.limit(10).collect())


shape: (10, 132)
┌──────────┬─────────┬───────┬──────────┬───┬───────────┬──────────┬──────────┬───────────────┐
│ id       ┆ amostra ┆ foto  ┆ conceito ┆ … ┆ trat_foto ┆ trat_mes ┆ trat_ano ┆ trat_data_ref │
│ ---      ┆ ---     ┆ ---   ┆ ---      ┆   ┆ ---       ┆ ---      ┆ ---      ┆ ---           │
│ i64      ┆ str     ┆ str   ┆ i64      ┆   ┆ str       ┆ str      ┆ i16      ┆ date          │
╞══════════╪═════════╪═══════╪══════════╪═══╪═══════════╪══════════╪══════════╪═══════════════╡
│ 10000125 ┆ OOT     ┆ set22 ┆ 0        ┆ … ┆ set       ┆ 9        ┆ 2022     ┆ 2022-09-30    │
│ 10000177 ┆ DES     ┆ mai22 ┆ 0        ┆ … ┆ mai       ┆ 5        ┆ 2022     ┆ 2022-05-31    │
│ 10000360 ┆ OOT     ┆ set22 ┆ 0        ┆ … ┆ set       ┆ 9        ┆ 2022     ┆ 2022-09-30    │
│ 10001297 ┆ DES     ┆ jul22 ┆ 0        ┆ … ┆ jul       ┆ 7        ┆ 2022     ┆ 2022-07-31    │
│ 10002398 ┆ DES     ┆ dez21 ┆ 0        ┆ … ┆ dez       ┆ 12       ┆ 2021     ┆ 2021-12-31    │
│ 1000261  ┆ TTD     ┆ 

In [69]:
print(trat_grafo_base_atributos.columns)

lista_periodos = ["3mo", "6mo"]

trat_grafo_base_atributos_agg = trat_grafo_base_atributos.group_by_dynamic(
    pl.col("trat_data_ref").sort(), every="3mo"
).agg(
    (pl.last("trat_data_ref") + pl.lit(timedelta(days=-3 * 30)))
    .cast(pl.Date)
    .dt.month_end()
    .alias("inicio_intervalo"),
    pl.last("trat_data_ref").alias("fim_intervalo"),
    pl.count("id"),
)

print(trat_grafo_base_atributos_agg.collect())


['id', 'amostra', 'foto', 'conceito', 'var_1', 'var_2', 'var_11', 'var_12', 'var_13', 'var_14', 'var_15', 'var_16', 'var_17', 'var_18', 'var_19', 'var_20', 'var_31', 'var_32', 'var_33', 'var_34', 'var_35', 'var_36', 'var_37', 'var_38', 'var_39', 'var_40', 'var_51', 'var_52', 'var_53', 'var_54', 'var_55', 'var_56', 'var_57', 'var_58', 'var_59', 'var_60', 'var_61', 'var_62', 'var_63', 'var_64', 'var_65', 'var_66', 'var_67', 'var_68', 'var_69', 'var_70', 'var_71', 'var_72', 'var_73', 'var_74', 'var_75', 'var_76', 'var_77', 'var_78', 'var_79', 'var_80', 'var_81', 'var_83', 'var_84', 'var_85', 'var_86', 'var_88', 'var_89', 'var_90', 'var_91', 'var_93', 'var_94', 'var_95', 'var_96', 'var_97', 'var_98', 'var_99', 'var_100', 'var_101', 'var_102', 'var_103', 'var_104', 'var_105', 'var_106', 'var_107', 'var_108', 'var_109', 'var_110', 'var_111', 'var_197', 'var_198', 'var_199', 'var_200', 'var_201', 'var_202', 'var_203', 'var_204', 'var_205', 'var_206', 'var_207', 'var_208', 'var_209', 'var_210'

#### Tratamento de valores extremos (Avaliar corretude da Solução)

In [82]:
print(trat_grafo_base_atributos.columns)

for coluna in trat_grafo_base_atributos.select(cs.by_index(4, 130)).columns:
    limite_inferior = (
        trat_grafo_base_atributos.select(coluna).quantile(0.01).collect().to_dict()
    )
    limite_superior = (
        trat_grafo_base_atributos.select(coluna).quantile(0.99).collect().to_dict()
    )
    
    trat_grafo_base_atributos = trat_grafo_base_atributos.with_columns(
        coluna=pl.when(pl.col(coluna) < pl.lit(limite_inferior[coluna][0]))
        .then(pl.lit(limite_inferior[coluna][0]))
        .otherwise(pl.col(coluna))
    ).with_columns(
        coluna=pl.when(pl.col(coluna) > pl.lit(limite_superior[coluna][0]))
        .then(pl.lit(limite_superior[coluna][0]))
        .otherwise(pl.col(coluna))
    )


['id', 'amostra', 'foto', 'conceito', 'var_1', 'var_2', 'var_11', 'var_12', 'var_13', 'var_14', 'var_15', 'var_16', 'var_17', 'var_18', 'var_19', 'var_20', 'var_31', 'var_32', 'var_33', 'var_34', 'var_35', 'var_36', 'var_37', 'var_38', 'var_39', 'var_40', 'var_51', 'var_52', 'var_53', 'var_54', 'var_55', 'var_56', 'var_57', 'var_58', 'var_59', 'var_60', 'var_61', 'var_62', 'var_63', 'var_64', 'var_65', 'var_66', 'var_67', 'var_68', 'var_69', 'var_70', 'var_71', 'var_72', 'var_73', 'var_74', 'var_75', 'var_76', 'var_77', 'var_78', 'var_79', 'var_80', 'var_81', 'var_83', 'var_84', 'var_85', 'var_86', 'var_88', 'var_89', 'var_90', 'var_91', 'var_93', 'var_94', 'var_95', 'var_96', 'var_97', 'var_98', 'var_99', 'var_100', 'var_101', 'var_102', 'var_103', 'var_104', 'var_105', 'var_106', 'var_107', 'var_108', 'var_109', 'var_110', 'var_111', 'var_197', 'var_198', 'var_199', 'var_200', 'var_201', 'var_202', 'var_203', 'var_204', 'var_205', 'var_206', 'var_207', 'var_208', 'var_209', 'var_210'

#### Descritiva simples



#### Categoricação de covariáveis

1. qcut
2. cut

#### (Bônus) 

- Plotagem
[Seaborn](https://www.kaggle.com/code/themlphdstudent/cheat-sheet-seaborn-charts)

- Visualização
[NiceGUI](https://nicegui.io/documentation)